In [9]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import psycopg2
from psycopg2.extras import execute_values
from common.helper import connect_postgresql
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from common.const import (
    SCHEMA, BATCH_SIZE,
    TABLE_RAW_REVIEWS,
    VECTOR_DB_TABLE_REVIEW_EMBEDDINGS
)

# Create embedding tables

In [6]:
product_embedding_sql = f"""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS {SCHEMA}.{VECTOR_DB_TABLE_REVIEW_EMBEDDINGS} (
    parent_asin VARCHAR(20) PRIMARY KEY,
    embedding VECTOR(384)
);
"""
conn = connect_postgresql()
cur = conn.cursor()

cur.execute(product_embedding_sql)
conn.commit()

# Generate embeddings

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

conn = connect_postgresql()
cursor = conn.cursor()

cursor.execute(f"TRUNCATE TABLE {SCHEMA}.{VECTOR_DB_TABLE_REVIEW_EMBEDDINGS}")
conn.commit()

cursor.execute(f"""
    SELECT parent_asin, title, text
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    WHERE text IS NOT NULL
    LIMIT 10000
""")

rows = cursor.fetchall()

insert_query = f"""
INSERT INTO {SCHEMA}.{VECTOR_DB_TABLE_REVIEW_EMBEDDINGS} (parent_asin, embedding)
VALUES %s ON CONFLICT (parent_asin) DO NOTHING
"""

batch = []
for asin, title, text in tqdm(rows):
    emb = model.encode(f"{title}. {text}").tolist()
    batch.append((asin, emb))

    if len(batch) >= BATCH_SIZE:
        execute_values(cursor, insert_query, batch)
        conn.commit()
        batch.clear()

# Insert remaining rows
if batch:
    execute_values(cursor, insert_query, batch)
    conn.commit()

conn.close()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Query 1: Given a product ID (`parent_asin`), retrieve the N most similar products.

In [ ]:
sql = f"""
SELECT parent_asin,
       embedding <=> (
           SELECT embedding
           FROM {SCHEMA}.{VECTOR_DB_TABLE_REVIEW_EMBEDDINGS}
           WHERE parent_asin = 'B000068PBK'
       ) AS distance
FROM {SCHEMA}.{VECTOR_DB_TABLE_REVIEW_EMBEDDINGS}
ORDER BY distance
LIMIT 10;
"""

conn = connect_postgresql()
cursor = conn.cursor()

cursor.execute(sql)
rows = cursor.fetchall()
conn.commit()

rows

[('B000068PBK', 0.0),
 ('B000A23CQM', 0.273254611291128),
 ('B00009RB0Z', 0.340978662400802),
 ('B000068PBJ', 0.349975408611483),
 ('B000068PBL', 0.367442231400079),
 ('B000068PBR', 0.380834359501766),
 ('B000068PBN', 0.381596807980252),
 ('B09444WCR1', 0.42222302357348),
 ('B000PKNPES', 0.422311169573997),
 ('B0009MFUOA', 0.431845537644092)]